# Notebook 07: breaking the best detector, part 1: point anomalies (KDD99)

**Class 03_03, three parts, one per kind of anomaly.** Each part does the same three things on its own data:
(1) **pick** the best detector of classes 03_01 and 03_02 for that kind of anomaly, by the numbers; (2) **break** it,
the way an adversary would, in feature space and on our own models; (3) **defend** it and measure what the defence costs.

| part | anomaly | data | notebook |
|:--|:--|:--|:--|
| 1 | **point**: one row is far from the normal rows | KDD99 connections | this one |
| 2 | **contextual**: normal value, wrong moment | NAB series with a daily cycle | 08 |
| 3 | **collective**: each sensor looks fine, the group does not | water plant, 8 sensors | 09 |

**Threat model** (no threat model, no security claim). Everything runs on public data and on detectors we trained.

| | evasion (test time) | poisoning (training time) |
|:--|:--|:--|
| goal | an attack scored below the threshold, still the same attack | a target accepted as normal after some retrains |
| capability | change only the **mutable** features (volume, timing) | add a few points per round to the retraining window; they must pass themselves |
| knowledge | normal data only (*mimicry*), or one score per query (*black box*) | the current model's score (grey box) |

**Metrics.** *Evasion rate*: of the attacks the detector catches, the share that passes after the change. *Queries*:
scores the attacker asked for. *Rounds*: retrains until the target is accepted.

## Part 1: point anomalies on KDD99

In [ ]:
import os
import sys
import time

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl
from scipy.stats import chi2

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
import deeplib as dl
import robustlib as rl
import thresholds as th
from aasplot import BLUE, GOLD, GREEN, GREY, LIGHT, RED

an.table_config()
ap.setup("03")


sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
names = sp.prep.names
mut = np.array([n in rl.MUTABLE for n in names])
lo, hi = Xtr.min(0), Xtr.max(0)  # the attacker stays inside the range seen in normal traffic
Xatt, cat = Xte[y == 1], sp.test["category"].to_numpy()[y == 1]

## A. Pick the best detector

Eight detectors from classes 03_01 and 03_02 on the same split, each fitted on normal rows only, threshold at 1% false
alarms on normal validation rows. The best is the one with the highest AUPRC; the table also shows how many
parameters an attacker may have to understand (nothing here is secret: the attacker is assumed to know the family).

In [ ]:
AE = dl.AE(widths=(64, 32))
V = {d.name: rl.Victim.build(d, Xtr, Xva) for d in
     [an.Mahalanobis(), an.GMM(), an.KNNDist(), an.OCSVM(), an.IForest(), an.PCARecon(), AE, dl.DeepSVDD()]}  # fmt: skip
clean = pl.DataFrame(
    [{"detector": k, **an.evaluate(v.det.score(Xte), y, v.det.score(Xva))} for k, v in V.items()]
).sort("AUPRC", descending=True)
BEST = clean["detector"][0]
print(clean.select(["detector", "AUROC", "AUPRC", "recall", "precision", "F1"]))
print("best:", BEST)

## B. Break it (1): evasion has a price, on Example 1

Point B of notebook 00 sends a lot ($x_1=1$) and receives little ($x_2=-1$). To pass a Mahalanobis detector at 1%
false alarms ($d_M^2\le\chi^2_{2,0.99}=9.21$) while still receiving $x_2=-1$, how much can a session send? Solve
$(x_1^2-1.8x_1x_2+x_2^2)/0.19\le 9.21$ for $x_1$: a quadratic in $x_1$.

In [ ]:
t = chi2.ppf(0.99, 2)
x2 = -1.0
# x1^2 - 1.8 x2 x1 + x2^2 - 0.19 t <= 0
a, b, c = 1.0, -1.8 * x2, x2**2 - 0.19 * t
x1_max = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)
x1_min = (-b - np.sqrt(b * b - 4 * a * c)) / (2 * a)
print(f"threshold d^2 = {t:.2f}; with x2 = -1 the session may send at most x1 = {x1_max:.2f} (B had 1.00)")
print(
    f"the quadratic has roots {x1_min:.2f} and {x1_max:.2f}: feasible sessions have {x1_min:.2f} <= x1 <= {x1_max:.2f}"
)

In [ ]:
B = an.EX1_POINTS["B"]


def square_axes(ax, lim: float = 3.5, ticks=(-2, 0, 2)) -> None:
    """Equal-scale square axes of Example 1 (standardised log bytes sent and received)."""
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xlabel("$x_1$ log bytes sent")
    ax.set_ylabel("$x_2$ log bytes received")


def evasion_price(threshold: float, x2: float = -1.0) -> None:
    """Example 1: sessions on the line x2 = const that pass the Mahalanobis detector (d^2 <= threshold)."""
    X = an.example1(500, 0)
    c = x2**2 - 0.19 * threshold
    lo, hi = np.roots([1.0, -1.8 * x2, c])
    lo, hi = min(lo, hi), max(lo, hi)
    f, ax = ap.fig("col", 2.1)
    ax.scatter(X[:, 0], X[:, 1], s=3, color=GREY, alpha=0.5, lw=0)
    ax.plot(*ap.ellipse(an.EX1_MU, an.EX1_COV, threshold**0.5).T, color=BLUE, lw=1.3)
    ax.axhline(x2, color=LIGHT, lw=0.8)
    ax.plot([lo, hi], [x2, x2], color=GREEN, lw=3, solid_capstyle="butt")
    ax.scatter(*B, s=26, color=RED, marker="s", zorder=5)
    ax.annotate(
        "B: caught",
        B,
        (0.4, -3.05),
        color=RED,
        fontsize=8,
        ha="center",
        arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.6},
    )
    ax.text(-3.3, 3.2, f"green: passes ($x_1\\leq{hi:.2f}$)", color=GREEN, fontsize=8, ha="left", va="top")
    square_axes(ax)
    ap.save(f, "evasion_price")


evasion_price(float(t))

B can pass, but only by sending **less per session** ($x_1\le 0.35$ instead of 1 on the standardised log scale):
the detector does not make exfiltration impossible, it makes it **slower** and therefore longer and easier to catch
by other means. That is the honest meaning of "robust": it raises the attacker's cost.



## C. Break it (2): mimicry, borrow the volume and timing of a similar normal connection

For each caught attack, find the normal training row closest in the **fixed** features and copy its mutable
features (Wagner and Soto 2002 called this a mimicry attack). No query is needed.

In [ ]:
rows = []
for k, v in V.items():
    caught = v.alarm(Xatt)
    Xm = rl.mimicry(Xatt[caught], Xtr, mut)
    passed = ~v.alarm(Xm)
    rows.append({"detector": k, "caught": caught.mean(), "evasion (mutable only)": passed.mean(),
                 **{c: passed[cat[caught] == c].mean() if (cat[caught] == c).any() else None for c in an.CATEGORIES}})  # fmt: skip
mim = pl.DataFrame(rows)
print(mim)

In [ ]:
SHORT = {
    "GMM (-log-likelihood)": "GMM",
    "PCA reconstruction": "PCA recon.",
    "kNN distance": "kNN",
    "Deep SVDD": "Deep SVDD",
}


def short(name: str) -> str:
    return SHORT.get(name, name)


def mimicry(rows) -> None:
    """rows: dicts with detector, caught, `evasion (mutable only)`; two panels: share caught, share that evades after mimicry."""
    keep = ["Mahalanobis", "PCA reconstruction", AE.name, "GMM (-log-likelihood)", "Deep SVDD"]
    d = {r["detector"]: r for r in rows}
    f, axs = ap.fig("full", 1.15, ncols=2, sharey=True, gridspec_kw={"wspace": 0.08})
    ys = np.arange(len(keep))[::-1]
    for ax, key, col, title in (
        (axs[0], "caught", BLUE, "attacks caught"),
        (axs[1], "evasion (mutable only)", RED, "evade after mimicry (share of the caught)"),
    ):
        v = [d[k][key] for k in keep]
        ax.barh(ys, v, color=col, height=0.6)
        for y, val in zip(ys, v):
            ax.text(val + 0.01, y, f"{val:.3f}" if val < 0.2 else f"{val:.2f}", va="center", fontsize=7)
        ax.set(title=title, xlim=(0, 1.0 if key == "caught" else 0.3))
        ax.title.set_fontsize(7.5)
        ax.grid(axis="y", visible=False)
    axs[0].set_yticks(ys, [short(k) for k in keep])
    ap.save(f, "mimicry")


mimicry(mim.to_dicts())

**Reading.** Copying volume and timing evades few of the caught attacks (2% for Mahalanobis, 12% for Deep SVDD).
The detectors catch these attacks by what they **do** (SYN errors, rejected connections, failed logins, root
shells), not by how big or how long they are. Most evasions are r2l and u2r rows: those already look like normal
logins and sit near the threshold.

**Defender's lesson:** a detector is as robust as the features it relies on are hard to change. Features that the
attack *needs* (flags, error rates, privileged operations) are robust; volume and timing are cheap to fake.

## D. Break it (3): black-box search with a query budget

The attacker now gets a score per query and changes one mutable feature at a time (a random step, kept only if the
score goes down), inside the normal range, until the row passes or the budget runs out. The Isolation Forest is left
out (a query costs 10 ms, the search would take minutes). The last rows lift the constraint: **every** feature is
mutable (the row is then no longer the attack; this only shows what the constraint is worth).

In [ ]:
rows = []
for k in ["Mahalanobis", "GMM (-log-likelihood)", "kNN distance", "One-class SVM", "PCA reconstruction", AE.name]:
    v = V[k]
    caught = np.flatnonzero(v.alarm(Xatt))[:60]
    for budget in [50, 200, 1000]:
        t0 = time.time()
        res = [rl.random_search(v, Xatt[i], mut, lo, hi, budget=budget, seed=int(i)) for i in caught]
        ok = [r for r in res if r[1]]
        rows.append({"detector": k, "budget": budget, "evasion": len(ok) / len(res),
                     "queries (successes)": np.mean([r[2] for r in ok]) if ok else None,
                     "evaded categories": ",".join(sorted({cat[i] for i, r in zip(caught, res) if r[1]})),
                     "seconds": time.time() - t0})  # fmt: skip
for k in ["Mahalanobis", "PCA reconstruction", AE.name]:
    v = V[k]
    caught = np.flatnonzero(v.alarm(Xatt))[:60]
    res = [rl.random_search(v, Xatt[i], np.ones_like(mut), lo, hi, budget=1000, seed=int(i)) for i in caught]
    ok = [r for r in res if r[1]]
    rows.append(
        {
            "detector": k + " (all features)",
            "budget": 1000,
            "evasion": len(ok) / len(res),
            "queries (successes)": np.mean([r[2] for r in ok]) if ok else None,
            "evaded categories": ",".join(sorted({cat[i] for i, r in zip(caught, res) if r[1]})),
            "seconds": None,
        }
    )
bb = pl.DataFrame(rows)
print(bb)

In [ ]:
def blackbox(rows) -> None:
    """rows: dicts with detector, budget, evasion; `(all features)` rows are the unconstrained search at 1,000 queries."""
    keep = {"Mahalanobis": BLUE, "PCA reconstruction": GREEN, AE.name: GOLD, "One-class SVM": GREY}
    f, ax = ap.fig("full", 1.25)
    for name, col in keep.items():
        pts = sorted((r["budget"], r["evasion"]) for r in rows if r["detector"] == name)
        ax.plot(*zip(*pts), color=col, marker="o", ms=3.5, label=short(name))
        free = [r["evasion"] for r in rows if r["detector"] == f"{name} (all features)"]
        if free:
            ax.scatter([1000], free, marker="*", s=40, color=col, zorder=5)
    ax.annotate(
        "stars: all features free",
        (1000, 0.3),
        (60, 0.27),
        fontsize=7,
        color=GREY,
        arrowprops={"arrowstyle": "-", "color": GREY, "lw": 0.6},
    )
    ax.set(
        xscale="log",
        xlabel="queries per attack row",
        ylabel="share that evades",
        xticks=[50, 200, 1000],
        ylim=(-0.01, 0.34),
    )
    ax.set_xticklabels(["50", "200", "1,000"])
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
    ap.save(f, "blackbox")


blackbox(bb.to_dicts())

With the constraint, even 1,000 queries per row evade at most 5% (a few DoS rows); without it the same search
evades 15-30% of the rows (Mahalanobis 0.20, AE 0.15, PCA 0.30; a random one-feature-at-a-time search is a weak attacker:
a gradient attack on the AE would do better). The feasible region, not the attacker's cleverness, is the limit. Compare with the spam
module, where adding words was always allowed and evasion reached 100%.


In [ ]:
def point_tradeoff(clean, mim) -> None:
    """Clean AUPRC against the mimicry evasion rate of each detector: the best detector is not the most robust one."""
    d = {r["detector"]: r["evasion (mutable only)"] for r in mim.to_dicts()}
    f, ax = ap.fig("full", 1.7)
    for r in clean.to_dicts():
        best = r["detector"] == BEST
        ax.scatter(d[r["detector"]] * 100, r["AUPRC"], s=26 if best else 16, color=RED if best else BLUE, zorder=3)
        ax.annotate(r["detector"].replace(" reconstruction", "").replace(" (-log-likelihood)", ""), (d[r["detector"]] * 100, r["AUPRC"]),
                    (4, -9 if r["detector"] == "Isolation Forest" else 3), textcoords="offset points", fontsize=6.5, color=RED if best else "#2C3E50")  # fmt: skip
    ax.set(xlabel="mimicry evasion rate (%), only volume and timing changed", ylabel="clean AUPRC")
    ap.save(f, "point_tradeoff")


point_tradeoff(clean, mim)

**The best detector is not free of weaknesses.** The 2-layer AE leads the clean ranking (AUPRC 0.70 against 0.68 for
the GMM) and resists volume mimicry well (2.6% evasion; Mahalanobis 1.8% is the lowest, Deep SVDD 12.1% the highest):
here accuracy and this kind of robustness do not conflict. What breaks it is what the attacker is *allowed* to
change. With every feature free, a one-feature-at-a-time search evades 15% of the attacks it catches; and the next
two sections attack what the detector *depends on* (its threshold, its training data) instead of the detector.

## E. Break it (4): the threshold itself

Notebook 04 showed that ISODATA on the log error can choose the threshold without labels once attacks are frequent.
That makes the *unlabelled sample* part of the attack surface: whoever adds rows to it moves the cut. Here the
attacker appends **decoys**, rows with an enormous error (a scaled-up copy of an attack is enough), to a sample of
the best detector's scores that holds 30% genuine attacks. The decoys pull the upper mode away from the genuine
attacks and the cut follows it. The quantile rule uses clean normal validation rows and does not move.

In [ ]:
s_te = V[BEST].det.score(Xte)
ia, i0 = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
base = np.r_[s_te[ia], s_te[np.random.default_rng(0).choice(i0, int(len(ia) * 7 / 3), replace=False)]]  # 30% attacks
rng = np.random.default_rng(1)
rows = []
for n_decoy in [0, 25, 50, 100, 200, 400, 800]:
    sample = np.r_[base, 100 * s_te.max() * rng.uniform(0.8, 1.2, n_decoy)]
    for rule, t in (
        ("ISODATA, log error", th.isodata(sample, log=True)),
        ("quantile 99% (normal validation)", th.quantile(V[BEST].det.score(Xva), 0.01)),
    ):
        rows.append(
            {"decoys": n_decoy, "rule": rule, "threshold": t, "recall of genuine attacks": float((s_te[ia] > t).mean())}
        )
dec = pl.DataFrame(rows)
print(dec.pivot(on="rule", index="decoys", values="recall of genuine attacks"))

In [ ]:
def decoys(dec) -> None:
    """Recall of the genuine attacks when the attacker adds decoys to the unlabelled sample the threshold is computed on."""
    f, ax = ap.fig("full", 1.4)
    for rule, col, mk in (("ISODATA, log error", RED, "^"), ("quantile 99% (normal validation)", BLUE, "o")):
        d = dec.filter(pl.col("rule") == rule)
        ax.plot(d["decoys"], d["recall of genuine attacks"], color=col, marker=mk, ms=3.5, label=rule)
    ax.set(xlabel="decoy rows added to the unlabelled sample", ylabel="recall of genuine attacks", ylim=(0, 1.02))
    ax.legend(loc="lower left", fontsize=6.8)
    ap.save(f, "thr_decoys")


decoys(dec)

A threshold learned from unlabelled data is as exposed as the data: 800 decoys cut the recall of the genuine
attacks from 0.85 to 0.31, because the cut follows the upper mode away from them. The quantile of clean validation rows is immune, because the
attacker cannot write into the validation set. Rule: **protect the data that sets the threshold as you protect the
training data**, and prefer a threshold that comes from data you control.

## F. Break it (5): poisoning the normal model

A detector that is retrained on recent traffic to follow legitimate change can be taught a new normal, a little at a
time: the **boiling frog** (Rubinstein et al. 2009 on the PCA detector of backbone traffic; Kravchik and Shabtai
2020, "Can't boil this frog", on online-trained autoencoders). The mechanics are clearest on Example 1: each round the
defender refits on its window of the last 500 accepted points and sets the threshold at 1% false alarms; the attacker
adds 25 points just below the current threshold on the way to the target (Example 1's exfiltration session, scaled
to $(2,-2)$). We follow score(target) / threshold: above 1 the target alarms, below 1 it is accepted.

In [ ]:
X0 = an.example1(500, seed=0)
target = 2 * an.EX1_POINTS["B"]
print("target d^2 under the clean model:", round(float(an.Mahalanobis().fit(X0).score(target[None])[0]), 1))
runs = {
    "Mahalanobis (Ledoit-Wolf)": lambda: an.Mahalanobis(),
    "Mahalanobis (MCD, robust)": lambda: an.Mahalanobis(robust=True),
}
logs = {k: rl.boiling_frog(f, X0, target, rounds=20, per_round=25, keep=500, seed=0) for k, f in runs.items()}
rows = []
for k, log in logs.items():
    acc = next((r["round"] for r in log if r["target_ratio"] < 1), None)
    rows.append({"detector": k, "round accepted": acc, "poison points needed": None if acc is None else 25 * acc,
                 "share of the window": None if acc is None else min(1.0, 25 * acc / 500)})  # fmt: skip
print(pl.DataFrame(rows))

In [ ]:
def boiling_frog(logs: dict) -> None:
    """logs: {detector: [dict(round, target_ratio), ...]}; the target is accepted below the threshold line."""
    f, ax = ap.fig("full", 1.3)
    for (name, log), col, mk in zip(logs.items(), (BLUE, RED), "os"):
        ax.plot([r["round"] for r in log], [r["target_ratio"] for r in log], color=col, marker=mk, ms=3, label=name)
    ax.axhline(1, color=GREY, ls="--", lw=1)
    ax.text(20, 1.3, "below: the target is accepted", color=GREY, fontsize=7, ha="right")
    ax.set(yscale="log", xlabel="poisoning round (25 points each)", ylabel="score(target) / threshold")
    ax.legend(loc="lower left")
    ap.save(f, "boiling_frog")


boiling_frog(logs)

**Reading.** Both estimators accept the target after 8 or 9 rounds (200 to 225 poison points, 40-45% of the
window). The robust MCD does **not** help here: it discards *scattered* outliers (notebook 00, part C), but the poison
is a tight, patient cluster just inside the boundary, exactly the kind of dense region MCD keeps. Robust statistics
protect against noise and careless contamination, not against an attacker who adapts to the estimator.

## What the defender can watch (poisoning)

Poisoning must move the model, and a moving model is visible: track the centre of the training window between
retrains, and compare the newly accepted data with a frozen, trusted reference.

In [ ]:
def assess(name: str, score_tr, X_va, X_te, X_att, X_att_mimic) -> dict:
    """Clean metrics at 1% FPR and the mimicry evasion rate among the attacks caught."""
    s_va = score_tr(X_va)
    t = an.threshold(s_va)
    caught = score_tr(X_att) > t
    passed = score_tr(X_att_mimic[caught]) <= t
    return {
        "detector": name,
        **an.evaluate(score_tr(X_te), y, s_va),
        "caught": caught.mean(),
        "mimicry evasion": passed.mean(),
    }


Xmim = rl.mimicry(Xatt, Xtr, mut)

## G. Defend it (1): ensembles

* **Rank mean:** each member's score becomes its rank among normal validation scores (a number in [0, 1]); the
  ensemble score is the mean rank.
* **OR rule:** alarm if *any* member alarms; each member gets 1/m of the false-alarm budget (threshold at the
  $1-0.01/m$ quantile), so the ensemble still makes about 1% false alarms.

In [ ]:
members = [an.Mahalanobis(), an.GMM(), an.KNNDist(), an.PCARecon(), dl.DeepSVDD()]
for m in members:
    m.fit(Xtr)
rows = [assess(m.name, m.score, Xva, Xte, Xatt, Xmim) for m in members]
ens = rl.RankEnsemble(members, Xva)
ens.ref = [np.sort(m.score(Xva)) for m in members]  # members are already fitted
rows.append(assess("rank-mean ensemble", ens.score, Xva, Xte, Xatt, Xmim))
q = 1 - 0.01 / len(members)
ts = [np.quantile(m.score(Xva), q) for m in members]


def or_score(X):
    """max over members of score / own threshold: > 1 means at least one member alarms."""
    return np.max([m.score(X) / t for m, t in zip(members, ts)], axis=0)


rows.append(assess("OR ensemble", or_score, Xva, Xte, Xatt, Xmim))
print(pl.DataFrame(rows))

In [ ]:
def ensembles(rows) -> None:
    """rows: dicts with detector, AUPRC, `mimicry evasion`; two panels: clean AUPRC and evasion after mimicry."""
    keep = ["Mahalanobis", "GMM (-log-likelihood)", "Deep SVDD", "rank-mean ensemble", "OR ensemble"]
    d = {r["detector"]: r for r in rows}
    f, axs = ap.fig("full", 1.2, ncols=2, sharey=True, gridspec_kw={"wspace": 0.08})
    ys = np.arange(len(keep))[::-1]
    for ax, key, col, title, hi in (
        (axs[0], "AUPRC", BLUE, "clean AUPRC", 0.85),
        (axs[1], "mimicry evasion", RED, "evasion after mimicry", 0.2),
    ):
        v = [d[k][key] for k in keep]
        ax.barh(ys, v, color=[GREEN if "ensemble" in k else col for k in keep], height=0.6)
        for y, val in zip(ys, v):
            ax.text(val + 0.01, y, f"{val:.3f}", va="center", fontsize=7)
        ax.set(title=title, xlim=(0, hi))
        ax.title.set_fontsize(7.5)
        ax.grid(axis="y", visible=False)
    axs[0].set_yticks(ys, [short(k) for k in keep])
    ap.save(f, "ensembles")


ensembles(rows)

The rank-mean ensemble matches the best single detector without any tuning (AUPRC 0.69 against 0.70 for the
2-layer AE, AUROC 0.99): its members make different mistakes. Against mimicry it is *between* its members (0.065), not better than the most
robust one (Mahalanobis, 0.018): averaging ranks lets a large drop in one member's score pull the mean below the
threshold. The OR rule is stricter (0.043) but pays in false alarms when members disagree. An ensemble raises the
cost of attacks *tuned to one model*; it does not make weak features strong.

## H. Defend it (2): robust features

Fit the same detectors on the **fixed** features only (64 of 71): mimicry of volume and timing then cannot change
the score at all. The price is what the dropped features contributed on clean data.

In [ ]:
fx = ~mut
rows = []
for make in [an.Mahalanobis, an.GMM, an.PCARecon]:
    full, fixed = make().fit(Xtr), make().fit(Xtr[:, fx])
    rows.append(assess(full.name + " (all features)", full.score, Xva, Xte, Xatt, Xmim))
    rows.append(assess(fixed.name + " (fixed features)", fixed.score, Xva[:, fx], Xte[:, fx], Xatt[:, fx], Xmim[:, fx]))
print(pl.DataFrame(rows))

In [ ]:
def robust_features(rows) -> None:
    """rows: assess() rows named `<detector> (all features)` and `<detector> (fixed features)` with recall and mimicry evasion."""
    dets = list(dict.fromkeys(r["detector"].rsplit(" (", 1)[0] for r in rows))
    d = {r["detector"]: r for r in rows}
    f, ax = ap.fig("full", 1.3)
    x = np.arange(len(dets))
    for off, tag, col, lab in (
        (-0.19, "all features", BLUE, "all 71 features"),
        (0.19, "fixed features", GREEN, "64 fixed features"),
    ):
        v = [d[f"{k} ({tag})"]["recall"] for k in dets]
        bars = ax.bar(x + off, v, 0.36, color=col, label=lab, lw=0)
        for b, val in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, val + 0.02, f"{val:.2f}", ha="center", fontsize=7)
    ax.set_xticks(x, [f"{short(k)}\nevasion {d[f'{k} (all features)']['mimicry evasion']:.3f} to 0" for k in dets])
    ax.set(ylabel="recall at 1% false alarms", ylim=(0, 1.15))
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.25))
    ax.grid(axis="x", visible=False)
    ap.save(f, "robust_features")


robust_features(rows)

By construction mimicry evasion drops to 0. The clean price depends on the detector: PCA loses almost nothing
(recall 0.78 to 0.75), the Mahalanobis distance loses a quarter of its recall (0.68 to 0.52), because volume and
timing did carry part of the evidence. Choose the features by what an attacker can change, then measure the price.

## I. Defend it (3): vetted retraining

Notebook 08's attack, but each round the defender keeps only the new points that a **frozen reference** model
(fitted once on trusted data) also accepts at a looser threshold (0.1% false alarms). Legitimate slow change still
gets in; the poison has to stay inside the *original* normal region, so it cannot walk to the target.

In [ ]:
X0 = an.example1(500, seed=0)
target = 2 * an.EX1_POINTS["B"]
ref = an.Mahalanobis().fit(X0)
t_ref = an.threshold(ref.score(X0), fpr=0.001)


def frog(vet: bool) -> tuple[list[float], int]:
    """The boiling frog of notebook 08 on Example 1; with `vet` the defender keeps only the new points that the
    frozen reference model also accepts. Returns score(target) / threshold per round and the poison points kept."""
    rng = np.random.default_rng(0)
    data, ratios, kept = X0.copy(), [], 0
    for r in range(21):
        det = an.Mahalanobis().fit(data)
        t = an.threshold(det.score(data))
        ratios.append(float(det.score(target[None])[0] / t))
        c = data.mean(0)
        line = c + np.linspace(0, 1, 200)[:, None] * (target - c)
        ok = line[det.score(line) < 0.98 * t]
        poison = (ok[-1] if len(ok) else c) + rng.normal(0, 0.05, (25, 2))
        poison = poison[det.score(poison) < t]
        if vet:
            poison = poison[ref.score(poison) < t_ref]  # the vetting step
        kept += len(poison)
        data = np.vstack([data, poison])[-500:]
    return ratios, kept


ratios, kept = frog(vet=True)
plain, _ = frog(vet=False)
print("score(target)/threshold every 4 rounds:", [round(x, 2) for x in ratios[::4]])
print(f"poison points that passed the vetting: {kept} of {25 * 20}; target accepted: {min(ratios) < 1}")
print("without vetting, every 4 rounds:", [round(x, 2) for x in plain[::4]])

In [ ]:
def vetting(plain, vetted) -> None:
    """Lists of score(target) / threshold per round: without and with vetted retraining (Example 1, 20 rounds)."""
    f, ax = ap.fig("full", 1.3)
    ax.plot(range(len(plain)), plain, color=RED, marker="s", ms=3, label="retrain on everything")
    ax.plot(range(len(vetted)), vetted, color=GREEN, marker="o", ms=3, label="retrain on vetted points")
    ax.axhline(1, color=GREY, ls="--", lw=1)
    ax.text(20, 1.3, "below: the target is accepted", color=GREY, fontsize=7, ha="right")
    ax.set(yscale="log", xlabel="poisoning round (25 points each)", ylabel="score(target) / threshold")
    ax.legend(loc="lower left")
    ap.save(f, "vetting")


vetting(plain, ratios)

The vetting lets in 56 of the 500 poison points (those inside the frozen normal region); the model moves a little
(the ratio falls from 8.5 to 4.5) and then stops: the target is never accepted.

## J. Every defence has a price and an answer

| defence | stops | attacker's answer | cost |
|:--|:--|:--|:--|
| ensemble (OR) | attacks tuned to one detector | fool every member (more queries) | more compute; false alarms add up |
| robust features | cheap mimicry of volume and timing | change the attack itself (slower, noisier) | clean recall on attacks that differed in volume |
| vetted retraining | boiling frog | stay inside the frozen region (slow, small) | adapts slowly to real change |
| robust statistics (MCD) | scattered contamination | concentrated, patient poison (notebook 08) | compute; breakdown at 50% |

**No defence is final; each changes the price.** Evaluate every detector against a stated threat model, with
constraints that keep the attack an attack, and report both the clean metrics and the robustness. The best detector
of this part is not the most robust one; notebook 08 repeats the exercise for contextual anomalies.